In [ ]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt


In [ ]:
raw_path = 'data/raw'
processed_path = 'data/processed'
results_path = 'data/results'


# Specific Heat Modeling Dataset


In [ ]:
cp_moisture_model_df = pd.read_csv(f'{processed_path}/cp_moisture_model_df.csv')
cp_moisture_observed_df = pd.read_csv(f'{processed_path}/cp_moisture_observed_df.csv')
cp_moisture_volumetric_observed_df = pd.read_csv(f'{processed_path}/cp_moisture_volumetric_observed_df.csv')
temperature_cp_model_df = pd.read_csv(f'{processed_path}/temperature_cp_model_df.csv')
cp_from_cv_candidates_df = pd.read_csv(f'{processed_path}/cp_from_cv_candidates_df.csv')


In [ ]:
cp_master_model_df = pd.concat([
    cp_moisture_model_df,
    temperature_cp_model_df,
], ignore_index=True)

cp_master_with_cv_candidates_df = pd.concat([
    cp_master_model_df,
    cp_from_cv_candidates_df,
], ignore_index=True)

print('direct cp rows:', cp_master_model_df.shape)
print('direct cp + converted Cv rows:', cp_master_with_cv_candidates_df.shape)


# Recommended Modeling Subsets


In [ ]:
cp_from_moisture_primary_df = cp_moisture_volumetric_observed_df.reset_index(drop=True)

cp_from_moisture_observed_df = cp_moisture_observed_df.reset_index(drop=True)

cp_from_temperature_df = cp_master_model_df[
    cp_master_model_df['temperature_c'].notna()
].reset_index(drop=True)

cp_moisture_density_candidates_df = cp_from_cv_candidates_df[
    cp_from_cv_candidates_df['water_content'].notna()
    & cp_from_cv_candidates_df['bulk_density'].notna()
].reset_index(drop=True)

print('cp_from_moisture_primary_df', cp_from_moisture_primary_df.shape)
print('cp_from_moisture_observed_df', cp_from_moisture_observed_df.shape)
print('cp_from_temperature_df', cp_from_temperature_df.shape)
print('cp_moisture_density_candidates_df', cp_moisture_density_candidates_df.shape)


In [ ]:
feature_columns = [
    'water_content', 'water_content_basis', 'temperature_c',
    'bulk_density', 'soil_class', 'data_type', 'source'
]

cp_master_model_df[feature_columns + ['target_value']].head()


# Grouped Validation Recommendation


In [ ]:
summary_df = cp_master_with_cv_candidates_df.groupby(['dataset_family', 'source']).agg(
    n_rows=('target_value', 'size'),
    min_target=('target_value', 'min'),
    max_target=('target_value', 'max')
).reset_index()
summary_df


In [ ]:
print('Use source-aware validation, e.g. GroupKFold with groups = source.')
print('Primary moisture model: cp_from_moisture_primary_df using volumetric observed rows only.')
print('Primary moisture predictors: water_content + bulk_density.')
print('Secondary moisture comparison: cp_from_moisture_observed_df using mixed bases and soil metadata.')
print('Temperature model: cp_from_temperature_df using temperature_c + soil_class.')
print('Converted-Cv model: cp_moisture_density_candidates_df as an optional ablation.')


# Modeling Setup


In [ ]:
def build_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])

    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
        ]
    )


def evaluate_grouped_models(df, numeric_features, categorical_features, group_col='source', n_splits=3):
    model_df = df.copy()
    model_df = model_df.dropna(subset=['target_value']).reset_index(drop=True)

    X = model_df[numeric_features + categorical_features]
    y = model_df['target_value']

    if group_col in model_df.columns:
        groups = model_df[group_col].fillna('Unknown')
    else:
        groups = pd.Series(['Unknown'] * len(model_df))

    unique_groups = groups.nunique()
    validation_mode = 'grouped'
    split_iterator = None

    if unique_groups >= 2:
        n_splits = min(n_splits, unique_groups)
        if n_splits < 2:
            raise ValueError('Grouped validation requires at least 2 splits.')
        cv = GroupKFold(n_splits=n_splits)
        split_iterator = cv.split(X, y, groups=groups)
    else:
        validation_mode = 'ungrouped'
        n_splits = min(n_splits, len(model_df))
        if n_splits < 2:
            raise ValueError('Need at least 2 rows for fallback KFold validation.')
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        split_iterator = cv.split(X, y)

    preprocessor = build_preprocessor(numeric_features, categorical_features)
    models = {
        'ridge': Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', Ridge(alpha=1.0)),
        ]),
        'random_forest': Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', RandomForestRegressor(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                random_state=42,
            )),
        ]),
    }

    rows = []
    predictions = []
    split_list = list(split_iterator)
    for model_name, pipeline in models.items():
        for fold, (train_idx, test_idx) in enumerate(split_list, start=1):
            fitted = clone(pipeline)
            fitted.fit(X.iloc[train_idx], y.iloc[train_idx])
            y_pred = fitted.predict(X.iloc[test_idx])
            y_true = y.iloc[test_idx].to_numpy()
            rmse = mean_squared_error(y_true, y_pred) ** 0.5
            rows.append({
                'model': model_name,
                'fold': fold,
                'n_train': len(train_idx),
                'n_test': len(test_idx),
                'mae': mean_absolute_error(y_true, y_pred),
                'rmse': rmse,
                'r2': r2_score(y_true, y_pred),
                'validation_mode': validation_mode,
                'n_groups': unique_groups,
            })
            for idx, pred, true in zip(test_idx, y_pred, y_true):
                predictions.append({
                    'model': model_name,
                    'fold': fold,
                    'source': groups.iloc[idx],
                    'observed': true,
                    'predicted': pred,
                    'validation_mode': validation_mode,
                })

    metrics_df = pd.DataFrame(rows)
    summary_df = metrics_df.groupby(['model', 'validation_mode', 'n_groups'], as_index=False).agg(
        mean_mae=('mae', 'mean'),
        mean_rmse=('rmse', 'mean'),
        mean_r2=('r2', 'mean'),
    )
    predictions_df = pd.DataFrame(predictions)
    return metrics_df, summary_df, predictions_df


def evaluate_within_source_models(df, numeric_features, categorical_features, source_col='source', n_splits=3, min_rows=8):
    model_df = df.copy()
    model_df = model_df.dropna(subset=['target_value', source_col]).reset_index(drop=True)

    models = {
        'ridge': Ridge(alpha=1.0),
        'random_forest': RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            random_state=42,
        ),
    }

    rows = []
    predictions = []
    for source_name, source_df in model_df.groupby(source_col):
        if len(source_df) < min_rows:
            continue

        X = source_df[numeric_features + categorical_features]
        y = source_df['target_value']
        n_local_splits = min(n_splits, len(source_df))
        if n_local_splits < 2:
            continue

        cv = KFold(n_splits=n_local_splits, shuffle=True, random_state=42)
        preprocessor = build_preprocessor(numeric_features, categorical_features)

        split_list = list(cv.split(X, y))
        for model_name, base_model in models.items():
            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('model', base_model),
            ])
            for fold, (train_idx, test_idx) in enumerate(split_list, start=1):
                fitted = clone(pipeline)
                fitted.fit(X.iloc[train_idx], y.iloc[train_idx])
                y_pred = fitted.predict(X.iloc[test_idx])
                y_true = y.iloc[test_idx].to_numpy()
                rmse = mean_squared_error(y_true, y_pred) ** 0.5
                rows.append({
                    'source': source_name,
                    'model': model_name,
                    'fold': fold,
                    'n_train': len(train_idx),
                    'n_test': len(test_idx),
                    'mae': mean_absolute_error(y_true, y_pred),
                    'rmse': rmse,
                    'r2': r2_score(y_true, y_pred),
                    'validation_mode': 'within_source',
                })
                for idx, pred, true in zip(test_idx, y_pred, y_true):
                    predictions.append({
                        'source': source_name,
                        'model': model_name,
                        'fold': fold,
                        'observed': true,
                        'predicted': pred,
                        'validation_mode': 'within_source',
                    })

    metrics_df = pd.DataFrame(rows)
    summary_df = metrics_df.groupby(['source', 'model', 'validation_mode'], as_index=False).agg(
        mean_mae=('mae', 'mean'),
        mean_rmse=('rmse', 'mean'),
        mean_r2=('r2', 'mean'),
    )
    predictions_df = pd.DataFrame(predictions)
    return metrics_df, summary_df, predictions_df


# Model 1: Primary Moisture-Based Specific Heat


In [ ]:
moisture_primary_numeric_features = ['water_content', 'bulk_density']
moisture_primary_categorical_features = []

moisture_primary_metrics_df, moisture_primary_summary_df, moisture_primary_predictions_df = evaluate_grouped_models(
    cp_from_moisture_primary_df,
    numeric_features=moisture_primary_numeric_features,
    categorical_features=moisture_primary_categorical_features,
    group_col='source',
)

moisture_primary_summary_df


In [ ]:
moisture_primary_metrics_df


# Model 1b: Secondary Moisture Comparison


In [ ]:
moisture_secondary_numeric_features = ['water_content', 'bulk_density']
moisture_secondary_categorical_features = ['water_content_basis', 'soil_class', 'data_type']

moisture_secondary_metrics_df, moisture_secondary_summary_df, moisture_secondary_predictions_df = evaluate_grouped_models(
    cp_from_moisture_observed_df,
    numeric_features=moisture_secondary_numeric_features,
    categorical_features=moisture_secondary_categorical_features,
    group_col='source',
)

moisture_secondary_summary_df


In [ ]:
moisture_secondary_metrics_df


# Model 2: Temperature-Based Specific Heat


In [ ]:
temperature_numeric_features = ['temperature_c']
temperature_categorical_features = ['soil_class']

temperature_metrics_df, temperature_summary_df, temperature_predictions_df = evaluate_grouped_models(
    cp_from_temperature_df,
    numeric_features=temperature_numeric_features,
    categorical_features=temperature_categorical_features,
    group_col='source',
)

temperature_summary_df


In [ ]:
temperature_metrics_df


# Model 3: Moisture + Density Converted-Cv Ablation


In [ ]:
cv_candidate_numeric_features = ['water_content', 'bulk_density']
cv_candidate_categorical_features = ['water_content_basis', 'soil_class', 'data_type']

cv_candidate_metrics_df, cv_candidate_summary_df, cv_candidate_predictions_df = evaluate_grouped_models(
    cp_moisture_density_candidates_df,
    numeric_features=cv_candidate_numeric_features,
    categorical_features=cv_candidate_categorical_features,
    group_col='source',
)

cv_candidate_summary_df


In [ ]:
cv_candidate_metrics_df


# Within-Source Moisture Models


In [ ]:
moisture_within_metrics_df, moisture_within_summary_df, moisture_within_predictions_df = evaluate_within_source_models(
    cp_from_moisture_primary_df,
    numeric_features=moisture_primary_numeric_features,
    categorical_features=moisture_primary_categorical_features,
    source_col='source',
)

moisture_within_summary_df


In [ ]:
moisture_within_metrics_df


# Within-Source Temperature Models


In [ ]:
temperature_within_metrics_df, temperature_within_summary_df, temperature_within_predictions_df = evaluate_within_source_models(
    cp_from_temperature_df,
    numeric_features=temperature_numeric_features,
    categorical_features=temperature_categorical_features,
    source_col='source',
)

temperature_within_summary_df


In [ ]:
temperature_within_metrics_df


# Compare Transfer vs Within-Source


In [ ]:
comparison_rows = [
    {
        'task': 'moisture_primary_transfer',
        'best_rmse': moisture_primary_summary_df['mean_rmse'].min(),
        'best_r2': moisture_primary_summary_df['mean_r2'].max(),
    },
    {
        'task': 'moisture_primary_within_source',
        'best_rmse': moisture_within_summary_df['mean_rmse'].min() if not moisture_within_summary_df.empty else np.nan,
        'best_r2': moisture_within_summary_df['mean_r2'].max() if not moisture_within_summary_df.empty else np.nan,
    },
    {
        'task': 'temperature_transfer',
        'best_rmse': temperature_summary_df['mean_rmse'].min(),
        'best_r2': temperature_summary_df['mean_r2'].max(),
    },
    {
        'task': 'temperature_within_source',
        'best_rmse': temperature_within_summary_df['mean_rmse'].min() if not temperature_within_summary_df.empty else np.nan,
        'best_r2': temperature_within_summary_df['mean_r2'].max() if not temperature_within_summary_df.empty else np.nan,
    },
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_df


# Final Figures


In [ ]:
plot_df = comparison_df.copy()
plot_df['setting'] = plot_df['task'].str.replace('_', ' ')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(plot_df['setting'], plot_df['best_rmse'], color='steelblue')
axes[0].set_title('Best RMSE by Modeling Setting')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(plot_df['setting'], plot_df['best_r2'], color='darkorange')
axes[1].set_title('Best R^2 by Modeling Setting')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
best_moisture_model = moisture_within_summary_df.sort_values('mean_rmse').iloc[0]['model']
plot_df = moisture_within_predictions_df[
    moisture_within_predictions_df['model'] == best_moisture_model
].copy()

fig, ax = plt.subplots(figsize=(6, 6))
for source_name, group in plot_df.groupby('source'):
    ax.scatter(group['observed'], group['predicted'], label=source_name, alpha=0.8)

lims = [
    min(plot_df['observed'].min(), plot_df['predicted'].min()),
    max(plot_df['observed'].max(), plot_df['predicted'].max()),
]
ax.plot(lims, lims, 'k--', linewidth=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Observed Specific Heat (J/kg/K)')
ax.set_ylabel('Predicted Specific Heat (J/kg/K)')
ax.set_title(f'Within-Source Moisture Model: {best_moisture_model}')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
best_temperature_model = temperature_within_summary_df.sort_values('mean_rmse').iloc[0]['model']
plot_df = temperature_within_predictions_df[
    temperature_within_predictions_df['model'] == best_temperature_model
].copy()

fig, ax = plt.subplots(figsize=(6, 6))
for source_name, group in plot_df.groupby('source'):
    ax.scatter(group['observed'], group['predicted'], label=source_name, alpha=0.8)

lims = [
    min(plot_df['observed'].min(), plot_df['predicted'].min()),
    max(plot_df['observed'].max(), plot_df['predicted'].max()),
]
ax.plot(lims, lims, 'k--', linewidth=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Observed Specific Heat (J/kg/K)')
ax.set_ylabel('Predicted Specific Heat (J/kg/K)')
ax.set_title(f'Within-Source Temperature Model: {best_temperature_model}')
ax.legend()
plt.tight_layout()
plt.show()


# Final Summary Tables


In [ ]:
final_summary_df = pd.concat([
    moisture_within_summary_df.assign(task='moisture_within_source'),
    temperature_within_summary_df.assign(task='temperature_within_source'),
], ignore_index=True)

final_summary_df = final_summary_df[['task', 'source', 'model', 'mean_mae', 'mean_rmse', 'mean_r2']]
final_summary_df.sort_values(['task', 'mean_rmse'])


In [ ]:
paper_takeaways_df = pd.DataFrame([
    {
        'finding': 'Moisture transfer across sources',
        'result': f"Best RMSE = {comparison_df.loc[comparison_df['task'] == 'moisture_primary_transfer', 'best_rmse'].iloc[0]:.2f}; best R^2 = {comparison_df.loc[comparison_df['task'] == 'moisture_primary_transfer', 'best_r2'].iloc[0]:.3f}",
    },
    {
        'finding': 'Moisture within-source',
        'result': f"Best RMSE = {comparison_df.loc[comparison_df['task'] == 'moisture_primary_within_source', 'best_rmse'].iloc[0]:.2f}; best R^2 = {comparison_df.loc[comparison_df['task'] == 'moisture_primary_within_source', 'best_r2'].iloc[0]:.3f}",
    },
    {
        'finding': 'Temperature transfer across sources',
        'result': f"Best RMSE = {comparison_df.loc[comparison_df['task'] == 'temperature_transfer', 'best_rmse'].iloc[0]:.2f}; best R^2 = {comparison_df.loc[comparison_df['task'] == 'temperature_transfer', 'best_r2'].iloc[0]:.3f}",
    },
    {
        'finding': 'Temperature within-source',
        'result': f"Best RMSE = {comparison_df.loc[comparison_df['task'] == 'temperature_within_source', 'best_rmse'].iloc[0]:.2f}; best R^2 = {comparison_df.loc[comparison_df['task'] == 'temperature_within_source', 'best_r2'].iloc[0]:.3f}",
    },
])
paper_takeaways_df


# Export Modeling Results


In [ ]:
import os
os.makedirs(processed_path, exist_ok=True)

moisture_primary_summary_df.to_csv(
    f'{processed_path}/moisture_primary_model_summary.csv',
    index=False,
)
moisture_primary_metrics_df.to_csv(
    f'{processed_path}/moisture_primary_model_folds.csv',
    index=False,
)
moisture_primary_predictions_df.to_csv(
    f'{processed_path}/moisture_primary_model_predictions.csv',
    index=False,
)

moisture_secondary_summary_df.to_csv(
    f'{processed_path}/moisture_secondary_model_summary.csv',
    index=False,
)
moisture_secondary_metrics_df.to_csv(
    f'{processed_path}/moisture_secondary_model_folds.csv',
    index=False,
)
moisture_secondary_predictions_df.to_csv(
    f'{processed_path}/moisture_secondary_model_predictions.csv',
    index=False,
)

temperature_summary_df.to_csv(
    f'{processed_path}/temperature_model_summary.csv',
    index=False,
)
temperature_metrics_df.to_csv(
    f'{processed_path}/temperature_model_folds.csv',
    index=False,
)
temperature_predictions_df.to_csv(
    f'{processed_path}/temperature_model_predictions.csv',
    index=False,
)

cv_candidate_summary_df.to_csv(
    f'{processed_path}/cv_candidate_model_summary.csv',
    index=False,
)
cv_candidate_metrics_df.to_csv(
    f'{processed_path}/cv_candidate_model_folds.csv',
    index=False,
)
cv_candidate_predictions_df.to_csv(
    f'{processed_path}/cv_candidate_model_predictions.csv',
    index=False,
)

moisture_within_summary_df.to_csv(
    f'{processed_path}/moisture_within_model_summary.csv',
    index=False,
)
moisture_within_metrics_df.to_csv(
    f'{processed_path}/moisture_within_model_folds.csv',
    index=False,
)
moisture_within_predictions_df.to_csv(
    f'{processed_path}/moisture_within_model_predictions.csv',
    index=False,
)

temperature_within_summary_df.to_csv(
    f'{processed_path}/temperature_within_model_summary.csv',
    index=False,
)
temperature_within_metrics_df.to_csv(
    f'{processed_path}/temperature_within_model_folds.csv',
    index=False,
)
temperature_within_predictions_df.to_csv(
    f'{processed_path}/temperature_within_model_predictions.csv',
    index=False,
)

comparison_df.to_csv(
    f'{processed_path}/transfer_vs_within_source_summary.csv',
    index=False,
)
final_summary_df.to_csv(
    f'{processed_path}/final_within_source_summary.csv',
    index=False,
)
paper_takeaways_df.to_csv(
    f'{processed_path}/paper_takeaways_summary.csv',
    index=False,
)
